# PatchTST - Deep Learning Time Series Model
imports, set up wandb.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))

import numpy as np
import pandas as pd
import wandb

from neuralforecast import NeuralForecast
from neuralforecast.models import PatchTST

from src.preprocessing import load_raw, build_features, to_long_format, wmae
from src.config import WANDB_ENTITY, WANDB_PROJECT
from src.wandb_utils import safe_wandb_init, save_and_log_pipeline_artifact

np.random.seed(42)
print("W&B entity:", WANDB_ENTITY)
print("W&B project:", WANDB_PROJECT)

## 1. Load & Reshape Data for neuralforecast

In [2]:
train, test, features, stores = load_raw()
train_full = build_features(train, features, stores)
long_df = to_long_format(train_full)

print(long_df.shape)
long_df.head()

(421570, 4)


,unique_id,ds,y,IsHoliday
0,10_1,2010-02-05,40212.84,False
1,10_1,2010-02-12,67699.32,True
2,10_1,2010-02-19,49748.33,False
3,10_1,2010-02-26,33601.22,False
4,10_1,2010-03-05,36572.44,False


## 2. Filter Series & Configure Horizon

In [3]:
H = 39
INPUT_SIZE = 52
MIN_LEN = INPUT_SIZE + H

long_df_all = long_df.copy()  # unfiltered; Section 4's final fit refilters against INPUT_SIZE only
series_len = long_df.groupby("unique_id").size()
keep_ids = series_len[series_len >= MIN_LEN].index

print(f"series total: {series_len.shape[0]}")
print(f"series kept (len >= {MIN_LEN}): {len(keep_ids)}")
print(f"series dropped (too short): {series_len.shape[0] - len(keep_ids)}")

long_df = long_df[long_df["unique_id"].isin(keep_ids)].reset_index(drop=True)
print("rows after filtering:", long_df.shape[0])

with safe_wandb_init(
    run_name="PatchTST_Preprocessing",
    group="PatchTST",
    job_type="preprocessing",
    config={"horizon_h": H, "input_size": INPUT_SIZE, "min_series_length": MIN_LEN},
) as run:
    wandb.log({
        "series_total": series_len.shape[0],
        "series_kept": len(keep_ids),
        "series_dropped": series_len.shape[0] - len(keep_ids),
        "rows_kept": long_df.shape[0],
    })

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.


series total: 3331
series kept (len >= 91): 2900
series dropped (too short): 431
rows after filtering: 410069


wandb: Currently logged in as: abeku20 (abeku20-free-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


wandb: setting up run 46flrr9e


wandb: Tracking run with wandb version 0.28.0


...

wandb: uploading requirements.txt; uploading wandb-summary.json; uploading config.yaml; uploading wandb-metadata.json


wandb: uploading history steps 0-0, summary


wandb: 
wandb: Run history:
wandb:      rows_kept ▁
wandb: series_dropped ▁
wandb:    series_kept ▁
wandb:   series_total ▁
wandb: 
wandb: Run summary:
wandb:      rows_kept 410069
wandb: series_dropped 431
wandb:    series_kept 2900
wandb:   series_total 3331
wandb: 


wandb:  View run PatchTST_Preprocessing at: https://wandb.ai/gchal22-free-university-of-tbilisi-/store_sales_forecast/runs/46flrr9e
wandb:  View project at: https://wandb.ai/gchal22-free-university-of-tbilisi-/store_sales_forecast
wandb: Synced 4 W&B file(s), 0 media file(s), 0 artifact file(s) and 0 other file(s)


wandb: Find logs at: .\wandb\run-20260712_164534-46flrr9e\logs


## 3. Time-Based Cross-Validation

In [4]:
MAX_STEPS = 1200 

model = PatchTST(
    h=H,
    input_size=INPUT_SIZE,
    patch_len=16,
    stride=8,
    max_steps=MAX_STEPS,
    scaler_type="standard",
    random_seed=42,
    enable_progress_bar=False,
    logger=False,
)
nf = NeuralForecast(models=[model], freq="W-FRI")

cv_df = nf.cross_validation(df=long_df, n_windows=None, test_size=H)
cv_df = cv_df.merge(
    long_df[["unique_id", "ds", "IsHoliday"]], on=["unique_id", "ds"], how="left"
)
cv_df.head()

Seed set to 42


GPU available: False, used: False


TPU available: False, using: 0 TPU cores



  | Name         | Type              | Params | Mode 
-----------------------------------------------------------
0 | loss         | MAE               | 0      | train
1 | padder_train | ConstantPad1d     | 0      | train
2 | scaler       | TemporalNorm      | 0      | train
3 | model        | PatchTST_backbone | 430 K  | train
-----------------------------------------------------------
430 K     Trainable params
3         Non-trainable params
430 K     Total params
1.722     Total estimated model params size (MB)
90        Modules in train mode
0         Modules in eval mode


C:\Users\aluda bekurishvili\Desktop\home\projects\store_sales_forecasting\.venv\Lib\site-packages\pytorch_lightning\utilities\_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


`Trainer.fit` stopped: `max_steps=1200` reached.


GPU available: False, used: False


TPU available: False, using: 0 TPU cores


,unique_id,ds,cutoff,PatchTST,y,IsHoliday
0,10_1,2012-02-03,2012-01-27,34892.492188,36444.00,False
1,10_1,2012-02-10,2012-01-27,40783.589844,50434.11,True
2,10_1,2012-02-17,2012-01-27,46402.335938,74930.33,False
3,10_1,2012-02-24,2012-01-27,39576.496094,28751.57,False
4,10_1,2012-03-02,2012-01-27,34701.664062,30525.88,False


In [5]:
val_wmae = wmae(cv_df["y"], cv_df["PatchTST"], cv_df["IsHoliday"])
val_mae = (cv_df["y"] - cv_df["PatchTST"]).abs().mean()

print(f"Validation WMAE: {val_wmae:.2f}")
print(f"Validation MAE:  {val_mae:.2f}")

with safe_wandb_init(
    run_name="PatchTST_CV",
    group="PatchTST",
    job_type="cross_validation",
    config={"horizon_h": H, "input_size": INPUT_SIZE, "patch_len": 16, "stride": 8, "max_steps": MAX_STEPS},
) as run:
    wandb.log({"val_wmae": val_wmae, "val_mae": val_mae})

Validation WMAE: 1932.15
Validation MAE:  1878.45


wandb: setting up run w9z6ggn8


wandb: Tracking run with wandb version 0.28.0


wandb: Run data is saved locally in C:\Users\aluda bekurishvili\Desktop\home\projects\store_sales_forecasting\wandb\run-20260712_165306-w9z6ggn8
wandb: Run `wandb offline` to turn off syncing.


wandb: Syncing run PatchTST_CV


...

wandb: uploading wandb-summary.json; uploading config.yaml; uploading wandb-metadata.json; uploading requirements.txt


wandb: uploading history steps 0-0, summary


wandb: 
wandb: Run history:
wandb:  val_mae ▁
wandb: val_wmae ▁
wandb: 
wandb: Run summary:
wandb:  val_mae 1878.44819
wandb: val_wmae 1932.15324
wandb: 


wandb:  View run PatchTST_CV at: https://wandb.ai/gchal22-free-university-of-tbilisi-/store_sales_forecast/runs/w9z6ggn8
wandb:  View project at: https://wandb.ai/gchal22-free-university-of-tbilisi-/store_sales_forecast
wandb: Synced 4 W&B file(s), 0 media file(s), 0 artifact file(s) and 0 other file(s)


wandb: Find logs at: .\wandb\run-20260712_165306-w9z6ggn8\logs


## 4. Final Model Training (Full History) & Pipeline Export

In [6]:
final_keep_ids = series_len[series_len >= INPUT_SIZE].index
long_df_final = long_df_all[long_df_all["unique_id"].isin(final_keep_ids)].reset_index(drop=True)
print(f"series kept for final fit (len >= {INPUT_SIZE}): {len(final_keep_ids)} (vs {len(keep_ids)} for CV)")

final_model = PatchTST(
    h=H,
    input_size=INPUT_SIZE,
    patch_len=16,
    stride=8,
    max_steps=MAX_STEPS,
    scaler_type="standard",
    random_seed=42,
    enable_progress_bar=False,
    logger=False,
)
nf_final = NeuralForecast(models=[final_model], freq="W-FRI")
nf_final.fit(df=long_df_final)

SAVE_PATH = "models/patchtst_nf"
nf_final.save(path=SAVE_PATH, overwrite=True, save_dataset=True)
print("saved to", SAVE_PATH)

Seed set to 42


GPU available: False, used: False


TPU available: False, using: 0 TPU cores



  | Name         | Type              | Params | Mode 
-----------------------------------------------------------
0 | loss         | MAE               | 0      | train
1 | padder_train | ConstantPad1d     | 0      | train
2 | scaler       | TemporalNorm      | 0      | train
3 | model        | PatchTST_backbone | 430 K  | train
-----------------------------------------------------------
430 K     Trainable params
3         Non-trainable params
430 K     Total params
1.722     Total estimated model params size (MB)
90        Modules in train mode
0         Modules in eval mode


C:\Users\aluda bekurishvili\Desktop\home\projects\store_sales_forecasting\.venv\Lib\site-packages\pytorch_lightning\utilities\_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


series kept for final fit (len >= 52): 2991 (vs 2900 for CV)


`Trainer.fit` stopped: `max_steps=1200` reached.


saved to models/patchtst_nf


In [7]:
from src.dl_models import NeuralForecastPipeline

with safe_wandb_init(
    run_name="PatchTST_Final",
    group="PatchTST",
    job_type="final_training",
    config={"horizon_h": H, "input_size": INPUT_SIZE, "patch_len": 16, "stride": 8, "max_steps": MAX_STEPS},
) as run:
    wandb.log({"val_wmae": val_wmae, "final_series_kept": len(final_keep_ids)})

    save_and_log_pipeline_artifact(
        run,
        "patchtst-pipeline",
        dirs={"nf_model": SAVE_PATH},
        metadata={
            "architecture": "PatchTST",
            "metric": "WMAE",
            "val_wmae": val_wmae,
            "horizon_h": H,
            "input_size": INPUT_SIZE,
            "patch_len": 16,
            "stride": 8,
            "max_steps": MAX_STEPS,
            "uses_raw_test": True,
            "notes": "Univariate NeuralForecast model (no exogenous vars). Load via "
                     "src.dl_models.NeuralForecastPipeline.load(path, model_col='PatchTST').",
        },
        aliases=["latest", "candidate"],
    )
    print("Logged pipeline to W&B run:", run.id)

wandb: setting up run 8otolu57


wandb: Tracking run with wandb version 0.28.0


wandb: Run data is saved locally in C:\Users\aluda bekurishvili\Desktop\home\projects\store_sales_forecasting\wandb\run-20260712_170027-8otolu57
wandb: Run `wandb offline` to turn off syncing.


wandb: Syncing run PatchTST_Final


wandb:  View project at https://wandb.ai/gchal22-free-university-of-tbilisi-/store_sales_forecast


...

wandb: uploading artifact patchtst-pipeline


wandb: uploading history steps 0-0, summary


wandb: 
wandb: Run history:
wandb: final_series_kept ▁
wandb:          val_wmae ▁
wandb: 
wandb: Run summary:
wandb: final_series_kept 2991
wandb:          val_wmae 1932.15324
wandb: 


wandb:  View run PatchTST_Final at: https://wandb.ai/gchal22-free-university-of-tbilisi-/store_sales_forecast/runs/8otolu57
wandb:  View project at: https://wandb.ai/gchal22-free-university-of-tbilisi-/store_sales_forecast
wandb: Synced 4 W&B file(s), 0 media file(s), 5 artifact file(s) and 0 other file(s)


wandb: Find logs at: .\wandb\run-20260712_170027-8otolu57\logs
